This notebook contains Python and R code for reproducing the results in our paper on using a large language model as a filter to identify unsuitable sentences for automatic fill-in-the-blank cloze question generation:

***UPDATE***Dittel, J. S., Van Campenhout, R., & Johnson, B. G. (2025). Refining sentence selection for automatic cloze question generation with large language models. In _Proceedings of the Twelfth ACM Conference on Learning at Scale (L@S '25)_, Palermo, Italy. https://doi.org/10.1145/3698205.3733926, **pp. TODO: add page numbers when available**

Results are presented in the order they occur, organized by the paper's sections. For each result, an excerpt from the paper is given followed by code to compute the result from the data set provided. Example:

>While the original data set contained over 5.2 million sessions, this analysis uses textbooks from three publishers who granted permission for generative AI research. Filtering yields 1,305,957 sessions across 210,902 questions, 106,183 students, and 2,510 textbooks, remaining sufficiently large for robust modeling.

```len( sessions ), sessions.question_id.nunique(), sessions.student_id.nunique(), sessions.textbook_id.nunique()```

Please refer to the paper for additional context.

In [1]:
import pandas as pd

In [2]:
%load_ext rpy2.ipython

## Read dataset

In [3]:
sessions = pd.read_parquet( 'sessions.parquet' )
sessions.head()

,student_id,question_id,textbook_id,subject,thumbs_down,H1_first_correct,H2_cumulative_answered,H3_spelling_suggestion,H4_sentence_textrank_rank,H5_answer_tf_idf_rank,H6_answer_pos,H7_answer_log_probability,H8_answer_location,H9_feedback,H10_reviewed,H11_llm_rejected
0,26EFUDCGXGK2R2BMUA65,000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab...,9781071875674,Political Science,0,0,1,0,0.226576,0.011482,NOUN,-13.244523,9,outcome,0,0
1,AK53WK5B75TBJXS7MMSM,000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab...,9781071875674,Political Science,0,0,1,0,0.226576,0.011482,NOUN,-13.244523,9,outcome,0,0
2,VGEWC8NNTPF6ZNP2XGDM,000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab...,9781071875674,Political Science,0,0,1,0,0.226576,0.011482,NOUN,-13.244523,9,outcome,0,0
3,3G5CA6NTMHFZXTK4EQZK,00001bc7b94e02d63ccd61bafe8082b473c27ffb7c5d40...,9781506318134,Political Science,0,1,1,0,0.231308,0.014590,NOUN,-11.116641,6,common_answer,0,0
4,5XNZ4CZRKQREESC27MWR,00001bc7b94e02d63ccd61bafe8082b473c27ffb7c5d40...,9781506318134,Political Science,0,1,1,0,0.231308,0.014590,NOUN,-11.116641,6,common_answer,0,0


In [4]:
questions = pd.read_parquet( 'questions.parquet' )
questions.head()

,question_id,textbook_id,subject,students,thumbs_down,stem,answer,sentence,H11_llm_rejected
0,0000abde625a9e1e9cec5dbdae42055a3642b65efb6d09...,9781544356433,Social Science,75,0,"Debra worked at a McDonald's and, lacking othe...",childcare,"Debra worked at a McDonald's and, lacking othe...",1
1,00010f79d65317116ff1a972de55711223065dc1bf6571...,9781317217381,Psychology,1,0,The general failure of the ______ theories to ...,universalist,The general failure of the universalist theori...,0
2,000380395e0b1f9a2c9cadb955e0ebd6ca79142f59f4c6...,9781351754705,History,5,0,After Justinian officially ended the teaching ...,Athens,After Justinian officially ended the teaching ...,1
3,0007eb409efb5a2cd71ddc2a71310aa2f95651c09ffbae...,9781000800418,Business & Economics,1,0,"Regarding social ______, some people believe t...",sustainability,"Regarding social sustainability, some people b...",0
4,000b57422bac7cfdb2d729827f9ac3202d1c33bc124ce1...,9781351754705,History,1,0,Many voters may have been mindful of Rome's ne...,southern,Many voters may have been mindful of Rome's ne...,0


## 2 Methods

### 2.2 Modeling of Student Ratings

>While the original data set contained over 5.2 million sessions, this analysis uses textbooks from three publishers who granted permission for generative AI research. Filtering yields 1,305,957 sessions across 210,902 questions, 106,183 students, and 2,510 textbooks, remaining sufficiently large for robust modeling.

In [5]:
len( sessions ), sessions.question_id.nunique(), sessions.student_id.nunique(), sessions.textbook_id.nunique()

(1305957, 210902, 106183, 2510)

>Using the standard BISAC major subject heading classification [28] available for most of the textbooks, the top subject domains as a fraction of session data were Social Science (27.2%), Psychology (22.8%), and Political Science (14.2%).

In [6]:
sessions.subject.value_counts( normalize=True, dropna=False ).apply( lambda p: f'{p:.1%}' )

subject
Social Science                       27.2%
Psychology                           22.8%
Political Science                    14.2%
Business & Economics                 11.5%
Language Arts & Disciplines          11.2%
Education                             6.4%
Law                                   1.4%
Family & Relationships                1.1%
History                               0.6%
Medical                               0.6%
Science                               0.5%
Music                                 0.4%
Sports & Recreation                   0.3%
Nature                                0.3%
Technology & Engineering              0.3%
Performing Arts                       0.2%
Photography                           0.2%
Computers                             0.1%
Philosophy                            0.1%
None                                  0.1%
Health & Fitness                      0.1%
Art                                   0.1%
Architecture                          0.1%
Rel

>Each student often answered multiple questions, and each question could be answered by multiple students, violating independence assumptions. Mixed effects models help address such clustering. However, as in prior work, our data set was too large to fit random intercepts for both questions and students. In line with that work [19, 20], separate random intercepts models were tested for questions only versus students only, using the Bayesian Information Criterion (BIC) to guide selection as common for explanatory models [29]. The student intercepts model had substantially lower BIC, so we focus on those results.

In [7]:
%%R
library( arrow )
library( glmmTMB )

Some features are not enabled in this build of Arrow. Run `arrow_info()` for more information.
The repository you retrieved Arrow from did not include all of Arrow's features.
You can install a fully-featured version by running:
`install.packages('arrow', repos = 'https://apache.r-universe.dev')`.

Attaching package: ‘arrow’

The following object is masked from ‘package:utils’:

    timestamp



In [8]:
%%R
sessions <- read_parquet( 'sessions.parquet' )
str( sessions )

Classes ‘tbl_df’, ‘tbl’ and 'data.frame':	1305957 obs. of  16 variables:
 $ student_id               : chr  "26EFUDCGXGK2R2BMUA65" "AK53WK5B75TBJXS7MMSM" "VGEWC8NNTPF6ZNP2XGDM" "3G5CA6NTMHFZXTK4EQZK" ...
 $ question_id              : chr  "000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab888522e6de2ced78c8" "000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab888522e6de2ced78c8" "000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab888522e6de2ced78c8" "00001bc7b94e02d63ccd61bafe8082b473c27ffb7c5d4073211fa78cc9318134" ...
 $ textbook_id              : chr  "9781071875674" "9781071875674" "9781071875674" "9781506318134" ...
 $ subject                  : chr  "Political Science" "Political Science" "Political Science" "Political Science" ...
 $ thumbs_down              : int  0 0 0 0 0 0 0 0 0 0 ...
 $ H1_first_correct         : int  0 0 0 1 1 0 1 1 0 1 ...
 $ H2_cumulative_answered   : int  1 1 1 1 1 1 1 1 1 1 ...
 $ H3_spelling_suggestion   : int  0 0 0 0 0 0 0 0 1 0 ...
 $ H4_sentence_textrank_rank: n

Question intercepts model for comparison of BIC with student intercepts model used in this work (Table 2).

In [9]:
%%R
model <- glmmTMB( thumbs_down ~ H1_first_correct
                              + H2_cumulative_answered
                              + H3_spelling_suggestion
                              + H4_sentence_textrank_rank
                              + H5_answer_tf_idf_rank
                              + H6_answer_pos
                              + H7_answer_log_probability
                              + H8_answer_location
                              + H9_feedback
                              + H10_reviewed
                              + H11_llm_rejected
                              + (1|question_id),
                              family=binomial(link=logit), data=sessions )
BIC( model )

[1] 28393.37


## 3 Results and Discussion

>Of the 210,902 unique questions, the LLM flagged 15,121 (7.17%) as unsuitable, spanning 83,629 sessions (6.40% of the data set).

In [10]:
print( f'{questions.H11_llm_rejected.sum()} {questions.H11_llm_rejected.mean():.2%}' )
print( f'{sessions.H11_llm_rejected.sum()} {sessions.H11_llm_rejected.mean():.2%}' )

15121 7.17%
83629 6.40%


>An example is the following sentence from a criminal investigation textbook [33]: “First and foremost may simply be the nature and structure of the crimes and how the police typically respond to them.” Because TextRank relies on word embeddings and co-occurrences to gauge sentence importance, it likely prioritized this sentence for containing words such as “crime,” “police,” and “structure.” The LLM, however, judged the sentence too vague, referencing multiple broad ideas without specifying a particular testable fact or concept.

In [11]:
idx = 81396
row = questions.loc[ idx ]
print( row.sentence )
row.to_frame()

First and foremost may simply be the nature and structure of the crimes and how the police typically respond to them.


,81396
question_id,c0c686f23722fb3abe7caadd85743bde1545f59ec561d9...
textbook_id,9781544395685
subject,Social Science
students,16
thumbs_down,1
stem,First and foremost may simply be the nature an...
answer,typically
sentence,First and foremost may simply be the nature an...
H11_llm_rejected,1


>Among the 1,305,957 sessions, thumbs down occurred in 2,407 (1.84 per 1,000). Notably, questions from LLM-rejected sentences had 4.91 thumbs down per 1,000 sessions, versus 1.63 for non-rejected, a threefold difference.

In [12]:
print( f'{sessions.thumbs_down.sum()} {sessions.thumbs_down.mean() * 1000:.2f}' )

2407 1.84


In [13]:
sessions.groupby( 'H11_llm_rejected' ).thumbs_down.mean().round( 5 ) * 1000

H11_llm_rejected
0    1.63
1    4.91
Name: thumbs_down, dtype: float64

>Table 2 summarizes the regression results, with odds ratios (exponentiated coefficients) used to gauge effect size from very small to large as described in Cohen [34].

Students intercepts model including H1–H11.

In [14]:
%%R
model <- glmmTMB( thumbs_down ~ H1_first_correct
                              + H2_cumulative_answered
                              + H3_spelling_suggestion
                              + H4_sentence_textrank_rank
                              + H5_answer_tf_idf_rank
                              + H6_answer_pos
                              + H7_answer_log_probability
                              + H8_answer_location
                              + H9_feedback
                              + H10_reviewed
                              + H11_llm_rejected
                              + (1|student_id),
                              family=binomial(link=logit), data=sessions )
summary( model )

 Family: binomial  ( logit )
Formula:          
thumbs_down ~ H1_first_correct + H2_cumulative_answered + H3_spelling_suggestion +  
    H4_sentence_textrank_rank + H5_answer_tf_idf_rank + H6_answer_pos +  
    H7_answer_log_probability + H8_answer_location + H9_feedback +  
    H10_reviewed + H11_llm_rejected + (1 | student_id)
Data: sessions

      AIC       BIC    logLik  deviance  df.resid 
  22801.1   23006.5  -11383.6   22767.1   1305940 

Random effects:

Conditional model:
 Groups     Name        Variance Std.Dev.
 student_id (Intercept) 100.6    10.03   
Number of obs: 1305957, groups:  student_id, 106183

Conditional model:
                            Estimate Std. Error z value Pr(>|z|)    
(Intercept)               -1.173e+01  2.316e-01  -50.66  < 2e-16 ***
H1_first_correct          -1.209e+00  5.782e-02  -20.92  < 2e-16 ***
H2_cumulative_answered    -2.308e-03  6.985e-04   -3.30 0.000953 ***
H3_spelling_suggestion    -5.301e-01  1.511e-01   -3.51 0.000450 ***
H4_sentence_t

>Controlling for previously known causal factors, H11_llm_rejected has an odds ratio of 3.41, meaning questions from rejected sentences have more than triple the odds of a thumbs down, a medium effect size.

In [15]:
%%R
exp( fixef( model )$cond )

              (Intercept)          H1_first_correct    H2_cumulative_answered 
             8.026083e-06              2.983823e-01              9.976949e-01 
   H3_spelling_suggestion H4_sentence_textrank_rank     H5_answer_tf_idf_rank 
             5.885589e-01              2.418461e+00              2.172276e+00 
         H6_answer_posADV         H6_answer_posNOUN        H6_answer_posPROPN 
             3.840445e+00              1.267669e+00              1.870444e+00 
        H6_answer_posVERB H7_answer_log_probability        H8_answer_location 
             2.003485e+00              1.094643e+00              9.887769e-01 
       H9_feedbackcontext        H9_feedbackoutcome              H10_reviewed 
             1.062467e+00              1.258398e+00              9.555153e-01 
         H11_llm_rejected 
             3.411189e+00 


>Adding H11 to a baseline model (H1–H10) reduces BIC by 200 points (23,227.0 to 23,006.5), confirming its explanatory value.

Student intercepts model including only H1–H10 to verify BIC = 23,227.0. BIC for full model (23,006.5) is seen above.

In [16]:
%%R
model <- glmmTMB( thumbs_down ~ H1_first_correct
                              + H2_cumulative_answered
                              + H3_spelling_suggestion
                              + H4_sentence_textrank_rank
                              + H5_answer_tf_idf_rank
                              + H6_answer_pos
                              + H7_answer_log_probability
                              + H8_answer_location
                              + H9_feedback
                              + H10_reviewed
                              + (1|student_id),
                              family=binomial(link=logit), data=sessions )
BIC( model )

[1] 23227.04
